# A Socio-Economic Model of the Housing Market
## Model Documentation: Pseudo-Code and UML Diagram

This notebook documents the structure of the agent-based model defined in `SocEconHousing1_abm.py`.

---
## 1. Model Overview

The model is a **discrete-time stochastic agent-based model** simulating the socio-economic segregation of a city. Two types of agents interact on a toroidal grid:

- **Households** are mobile agents with heterogeneous income and social status. They choose the housing unit that maximises their residential utility subject to a budget constraint.
- **Landlords** are stationary agents, one per grid cell, that own housing units. They update housing quality in response to neighbourhood rent trends and set rents based on the city-wide income-utility distribution.

The model is a **two-sided market**: landlords set rents based on the demand side (income distribution), and households choose locations based on the supply side (utility and rent). Neither side has complete information — landlords observe only neighbourhood rents and status, households observe only current rents and utilities.

---
## 2. Parameters

| Parameter | Symbol | Description |
|---|---|---|
| `size` | — | Grid side length (city = size × size cells) |
| `density` | — | Share of cells occupied by households |
| `vision` | — | Moore neighbourhood radius (landlord and household perception) |
| `distribution` | α | Shape parameter of the Beta distribution for income and quality |
| `r_correlation` | r | Income–status correlation |
| `a_preferences` | a | Status weight in landlord utility (Cobb-Douglas exponent) |
| `b_inertia` | b | Housing quality inertia (path dependence) |
| `turnover` | — | Share of households replaced per period |
| `max_time` | — | Number of simulation periods |

---
## 3. Simplified Pseudo-Code

### Setup

```
CREATE one housing unit (landlord) per grid cell
    housing_quality, utility, rent ~ Beta distribution

PLACE households on a random subset of cells  (share = density)
    income ~ Beta distribution
    status ~ correlated with income

COMPUTE the neighbourhood of each housing unit  (Moore, radius = vision, torus)
```

### Each Time Step

```
FOR each housing unit:
    UPDATE housing quality toward the average rent in its neighbourhood
    COMPUTE utility  =  neighbourhood avg. status^a  ×  housing quality^(1−a)

FOR each housing unit:
    SET rent  =  75th percentile income among households competing for
                 units of equal or lower utility  (city-wide)

FOR each household  (in random order):
    IF any affordable unit exists  (rent ≤ income):
        MOVE to the unit with the highest utility
    ELSE:
        MOVE to the cheapest available unit
```

---
## 4. Extensive Pseudo-Code

### Initialization

```
FOR each cell (x, y) in size × size grid:
    CREATE Landlord at (x, y)
        housing_quality ~ Beta(α, 2.5α)
        utility ← housing_quality
        rent    ← housing_quality

FOR each randomly sampled (density × size²) landlord positions:
    CREATE Household at (x, y)
        income ~ Beta(α, 2.5α)
        status ← (1 − r) × Beta(α, 2.5α) + r × income

FOR each landlord:
    COMPUTE Moore neighbourhood within radius vision (on torus)
    UPDATE household variables (hh_income, hh_status, empty)
```

### Simulation Loop (t = 0, 1, ..., max_time)

```
STEP 1 — LANDLORD INVESTMENT:
    FOR each landlord l:
        mean_rent_nb  ← mean(rent of l's neighbours)
        l.housing_quality ← b × l.housing_quality + (1−b) × mean_rent_nb

        mean_status_nb ← mean(hh_status of occupied neighbours)  [0 if none]
        l.utility ← mean_status_nb^a × l.housing_quality^(1−a)

STEP 2 — RENT SETTING:
    Build city-wide table of (utility, hh_income) pairs
    FOR each landlord l:
        competition ← incomes of households in units with utility ≤ l.utility
        IF competition not empty:
            l.rent ← 75th percentile of competition
        ELSE:
            l.rent ← city-wide minimum income

STEP 3 — HOUSEHOLD MOVEMENT (random order):
    SHUFFLE households
    FOR each household h:
        choice_set ← vacant units ∪ {h's current unit}
        budget_set ← {u ∈ choice_set : u.rent ≤ h.income}
        IF budget_set not empty:
            h moves to argmax(utility) in budget_set
        ELSE:
            h moves to argmin(rent) in choice_set
        UPDATE spatial index and affected landlords' household variables

STEP 4 — DATA RECORDING:
    Store (x, y, housing_quality, utility, rent, hh_id, hh_income, hh_status, t)

STEP 5 — POPULATION DYNAMICS (if turnover > 0):
    REMOVE turnover × N_hh randomly selected households
    UPDATE affected landlords' household variables
    ADD turnover × N_hh new households to random vacant positions
    UPDATE affected landlords' household variables

    t ← t + 1
```

### Key Equations

| Equation | Description |
|---|---|
| $\text{income}_i \sim \text{Beta}(\alpha,\, 2.5\alpha)$ | Household income distribution |
| $\text{status}_i = (1-r)\cdot\text{Beta}(\alpha,\,2.5\alpha) + r\cdot\text{income}_i$ | Status, correlated with income |
| $q_l^t = b\, q_l^{t-1} + (1-b)\,\overline{\text{rent}}_{N(l)}^{t-1}$ | Housing quality (inertia + neighbourhood rents) |
| $u_l = \bar{s}_{N(l)}^{\,a}\cdot q_l^{\,1-a}$ | Landlord utility (Cobb-Douglas: status × quality) |
| $r_l = P_{75}\{\text{income}_i : u_i \leq u_l\}$ | Rent (75th pct. of competing incomes) |

---
## 4. UML Class Diagram

The diagram below shows the class structure of the model. Run the cell below to render it.

In [ ]:
from IPython.display import display, HTML

diagram = """
classDiagram

    class SocEconHousing {
        +int size
        +float density
        +int vision
        +int max_time
        +float turnover
        +int time
        +List landlords
        +List households
        +Dict landlord_by_pos
        +Dict household_by_pos
        +DataFrame utility_income_df
        +run()
        +population_dynamics()
        +report() DataFrame
        +update_household_position()
    }

    class Landlord {
        +Tuple pos
        +float housing_quality
        +float utility
        +float rent
        +bool empty
        +float hh_income
        +float hh_status
        +List nb_ll
        +neighborhood() List
        +invest()
        +update_rent()
        +update_hhvars()
    }

    class Household {
        +int hh_id
        +Tuple pos
        +float income
        +float status
        +move()
    }

    SocEconHousing "1" *-- "size²" Landlord : contains
    SocEconHousing "1" *-- "N_hh" Household : contains
    Landlord "1" --> "vision²" Landlord : nb_ll (neighbourhood)
    Household "1" --> "1" Landlord : occupies
"""

html = f"""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<script>mermaid.initialize({{startOnLoad: true, theme: 'default'}});</script>
<div class="mermaid" style="font-size:14px;">{diagram}</div>
"""
display(HTML(html))

### UML Diagram (Mermaid source)

If the rendered diagram above does not display (e.g. without internet access), copy the Mermaid source below into [mermaid.live](https://mermaid.live) to render and export it.

```mermaid
classDiagram

    class SocEconHousing {
        +int size
        +float density
        +int vision
        +int max_time
        +float turnover
        +int time
        +List landlords
        +List households
        +Dict landlord_by_pos
        +Dict household_by_pos
        +DataFrame utility_income_df
        +run()
        +population_dynamics()
        +report() DataFrame
        +update_household_position()
    }

    class Landlord {
        +Tuple pos
        +float housing_quality
        +float utility
        +float rent
        +bool empty
        +float hh_income
        +float hh_status
        +List nb_ll
        +neighborhood() List
        +invest()
        +update_rent()
        +update_hhvars()
    }

    class Household {
        +int hh_id
        +Tuple pos
        +float income
        +float status
        +move()
    }

    SocEconHousing "1" *-- "size²" Landlord : contains
    SocEconHousing "1" *-- "N_hh" Household : contains
    Landlord "1" --> "vision²" Landlord : nb_ll (neighbourhood)
    Household "1" --> "1" Landlord : occupies
```

---
## 5. Agent Interaction Summary

| Actor | Decision | Criterion |
|---|---|---|
| Landlord | Investment | Adjust quality toward neighbourhood avg. rent (inertia `b`) |
| Landlord | Utility | Cobb-Douglas: neighbourhood status^a × quality^(1−a) |
| Landlord | Rent | 75th pct. income of households competing for ≤ this utility |
| Household | Location | Max utility within budget; cheapest unit if nothing affordable |